# Qwen2.5 MACCROBAT Evaluation

This notebook evaluates the local Qwen2.5 clinical NER Ollama workflow on MACCROBAT. It does not modify the existing Qwen extraction notebook or script. The helper functions live in `qwen25_maccrobat_eval.py`.

## Requirements

- Ollama must be running locally.
- The `qwen2.5:14b-instruct` model should be pulled in Ollama.
- Use either the Hugging Face `datasets` package or a local `data/maccrobat/data.jsonl` file.

Start with a small `MAX_DOCUMENTS` value because Qwen 14B can be slow over full clinical notes.

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "llm":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

LLM_DIR = PROJECT_ROOT / "models" / "llm"
if str(LLM_DIR) not in sys.path:
    sys.path.insert(0, str(LLM_DIR))

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "qwen25-maccrobat-evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT, OUTPUT_DIR

(PosixPath('/Users/kabilan/Documents/DL Lab/Deep-learning-Lab'),
 PosixPath('/Users/kabilan/Documents/DL Lab/Deep-learning-Lab/outputs/qwen25-maccrobat-evaluation'))

In [2]:
from collections import Counter, defaultdict
import importlib
import json

import pandas as pd

import qwen25_clinical_ner_ollama
import qwen25_maccrobat_eval

qwen25_clinical_ner_ollama = importlib.reload(qwen25_clinical_ner_ollama)
qwen25_maccrobat_eval = importlib.reload(qwen25_maccrobat_eval)

from qwen25_clinical_ner_ollama import ALLOWED_LABELS, MODEL_NAME
from qwen25_maccrobat_eval import (
    DEFAULT_DATASET,
    extract_entities_for_maccrobat,
    gold_spans_from_bio,
    load_json,
    load_maccrobat,
    prf,
    save_evaluation_outputs,
    score_document,
)

{
    "model": MODEL_NAME,
    "clinical_module": qwen25_clinical_ner_ollama.__file__,
    "eval_module": qwen25_maccrobat_eval.__file__,
    "label_count": len(ALLOWED_LABELS),
    "first_labels": ALLOWED_LABELS[:15],
}


{'model': 'qwen2.5:14b-instruct',
 'clinical_module': '/Users/kabilan/Documents/DL Lab/Deep-learning-Lab/models/llm/qwen25_clinical_ner_ollama.py',
 'eval_module': '/Users/kabilan/Documents/DL Lab/Deep-learning-Lab/models/llm/qwen25_maccrobat_eval.py',
 'label_count': 41,
 'first_labels': ['Activity',
  'Administration',
  'Age',
  'Area',
  'Biological_attribute',
  'Biological_structure',
  'Clinical_event',
  'Color',
  'Coreference',
  'Date',
  'Detailed_description',
  'Diagnostic_procedure',
  'Disease_disorder',
  'Distance',
  'Dosage']}

## MACCROBAT Labels

The LLM uses the same predefined MACCROBAT labels as the dataset, so no entity-label mapping is needed.


In [3]:
pd.DataFrame({"label": ALLOWED_LABELS})


,label
0,Activity
1,Administration
2,Age
3,Area
4,Biological_attribute
5,Biological_structure
6,Clinical_event
7,Color
8,Coreference
9,Date


## Configuration

If you have `data/maccrobat/data.jsonl`, set `LOCAL_JSONL` to that path. Otherwise leave it as `None` and the notebook will use Hugging Face `datasets`.

In [4]:
DATASET_NAME = DEFAULT_DATASET
SPLIT = "train"
MAX_DOCUMENTS = None
OLLAMA_MODEL = MODEL_NAME
OLLAMA_TIMEOUT = 900

candidate_jsonl = PROJECT_ROOT / "data" / "maccrobat" / "data.jsonl"
LOCAL_JSONL = candidate_jsonl if candidate_jsonl.exists() else None

{
    "dataset_name": DATASET_NAME,
    "split": SPLIT,
    "max_documents": MAX_DOCUMENTS,
    "local_jsonl": str(LOCAL_JSONL) if LOCAL_JSONL else None,
    "ollama_model": OLLAMA_MODEL,
}

{'dataset_name': 'ktgiahieu/maccrobat2018_2020',
 'split': 'train',
 'max_documents': None,
 'local_jsonl': None,
 'ollama_model': 'qwen2.5:14b-instruct'}

In [5]:
class DatasetArgs:
    dataset_name = DATASET_NAME
    local_jsonl = LOCAL_JSONL
    split = SPLIT


all_documents = load_maccrobat(DatasetArgs())
documents = all_documents if MAX_DOCUMENTS is None else all_documents[:MAX_DOCUMENTS]
len(documents), documents[0].keys() if documents else None

(400, dict_keys(['tokens', 'tags']))

## Inspect One Gold Document

This reconstructs text from MACCROBAT tokens and converts the BIO labels to span-level gold entities using the original MACCROBAT labels.


In [6]:
sample_text, sample_gold_entities = gold_spans_from_bio(
    documents[0]["tokens"],
    documents[0]["tags"],
)

print(sample_text[:1500])
pd.DataFrame(sample_gold_entities).head(25)

A 68 - year - old female nonsmoker , nondrinker with a medical history of hypertension presented with new - onset painless jaundice and pruritus , a three - month history of 9.9 kg weight loss and chronic diarrhea with four to five loose bowel movements per day . 
 Medications included vitamin D , amlodipine and eprosartan . 
 Physical examination was normal except for jaundice and muscle wasting . 
 Recent colonoscopy had been normal . 
 Total and direct bilirubin levels were 6.84 mg / dL ( 116.96 μmol / L ) and 9.18 mg / dL ( 156.98 μmol / L ) , respectively . 
 Other results included an international normalized ratio of 1.0 , alanine aminotransferase level 247 U / L ( normal <33 U / L ) , aspartate aminotransferase level 139 U / L ( normal < 32 U / L ) and alkaline phosphatase level 524 U / L ( normal 35 to 104 U / L ) . 
 Viral hepatitis serologies , and antimitochondrial antibody and anti - smooth muscle antibody tests were negative . 
 Her alpha - fetoprotein level was 2.4 ng / m

,text,label,start,end
0,68 - year - old,Age,2,17
1,female,Sex,18,24
2,nonsmoker,History,25,34
3,nondrinker,History,37,47
4,hypertension,Disease_disorder,74,86
5,presented,Clinical_event,87,96
6,jaundice,Sign_symptom,123,131
7,pruritus,Sign_symptom,136,144
8,three - month,Duration,149,162
9,weight loss,Sign_symptom,181,192


## Run Qwen on One Document

Run this cell first to confirm Ollama is working before evaluating multiple documents.

In [7]:
sample_result = extract_entities_for_maccrobat(sample_text, model=OLLAMA_MODEL, read_timeout=OLLAMA_TIMEOUT)
sample_predicted_entities = sample_result["entities"]
sample_scores = score_document(sample_gold_entities, sample_predicted_entities)

gold_labels = sorted({entity["label"] for entity in sample_gold_entities})
predicted_labels = sorted({entity["label"] for entity in sample_predicted_entities})

print(json.dumps({
    "scores": sample_scores,
    "metrics": prf(sample_scores["true_positive"], sample_scores["predicted"], sample_scores["gold"]),
    "gold_labels": gold_labels,
    "predicted_labels": predicted_labels,
}, indent=2))
pd.DataFrame(sample_predicted_entities).head(25)


{
  "scores": {
    "true_positive": 18,
    "predicted": 28,
    "gold": 74
  },
  "metrics": {
    "precision": 0.6428571428571429,
    "recall": 0.24324324324324326,
    "f1": 0.35294117647058826
  },
  "gold_labels": [
    "Age",
    "Biological_structure",
    "Clinical_event",
    "Date",
    "Detailed_description",
    "Diagnostic_procedure",
    "Disease_disorder",
    "Distance",
    "Duration",
    "History",
    "Lab_value",
    "Medication",
    "Sex",
    "Sign_symptom"
  ],
  "predicted_labels": [
    "Age",
    "Biological_structure",
    "Diagnostic_procedure",
    "Disease_disorder",
    "Duration",
    "Frequency",
    "Lab_value",
    "Mass",
    "Medication",
    "Personal_background",
    "Sex",
    "Sign_symptom"
  ]
}


,text,label,start,end
0,68 - year - old,Age,2,17
1,female,Sex,18,24
2,nonsmoker,Personal_background,25,34
3,nondrinker,Personal_background,37,47
4,hypertension,Disease_disorder,74,86
5,painless jaundice,Sign_symptom,114,131
6,pruritus,Sign_symptom,136,144
7,three - month history,Duration,149,170
8,9.9 kg,Mass,174,180
9,chronic diarrhea,Sign_symptom,197,213


## Evaluate Multiple Documents

This loops over `MAX_DOCUMENTS`, saves after every completed document, and resumes from `predictions.json` if the run is interrupted. A prediction is counted correct when the label matches, the spans overlap, and either the gold text words are a subset of the predicted text words or the predicted text words are a subset of the gold text words.


In [8]:
prediction_rows = load_json(OUTPUT_DIR / "predictions.json", [])
prediction_by_index = {int(row["document_index"]): row for row in prediction_rows}

for document_index, document in enumerate(documents):
    if document_index in prediction_by_index:
        print(f"Document {document_index}: already completed, skipping")
        continue

    text, gold_entities = gold_spans_from_bio(document["tokens"], document["tags"])
    result = extract_entities_for_maccrobat(text, model=OLLAMA_MODEL, read_timeout=OLLAMA_TIMEOUT)
    predicted_entities = result["entities"]
    scores = score_document(gold_entities, predicted_entities)

    prediction_by_index[document_index] = {
        "document_index": document_index,
        "text": text,
        "gold_entities": gold_entities,
        "predicted_entities": predicted_entities,
        **scores,
    }
    prediction_rows = list(prediction_by_index.values())
    summary, per_label_summary, document_rows = save_evaluation_outputs(OUTPUT_DIR, prediction_rows)

    metrics = prf(scores["true_positive"], scores["predicted"], scores["gold"])
    print(
        f"Document {document_index}: "
        f"f1={metrics['f1']:.3f}, predicted={scores['predicted']}, gold={scores['gold']}, "
        f"saved={len(prediction_rows)}/{len(documents)}"
    )

summary, per_label_summary, document_rows = save_evaluation_outputs(
    OUTPUT_DIR,
    list(prediction_by_index.values()),
)

summary


Document 0: already completed, skipping
Document 1: already completed, skipping
Document 2: f1=0.343, predicted=28, gold=71, saved=3/400
Document 3: f1=0.306, predicted=30, gold=42, saved=4/400
Document 4: f1=0.344, predicted=27, gold=66, saved=5/400
Document 5: f1=0.250, predicted=30, gold=106, saved=6/400
Document 6: f1=0.255, predicted=25, gold=30, saved=7/400
Document 7: f1=0.167, predicted=30, gold=30, saved=8/400
Document 8: f1=0.417, predicted=29, gold=67, saved=9/400
Document 9: f1=0.292, predicted=19, gold=29, saved=10/400
Document 10: f1=0.528, predicted=25, gold=47, saved=11/400
Document 11: f1=0.304, predicted=24, gold=68, saved=12/400
Document 12: f1=0.442, predicted=31, gold=46, saved=13/400
Document 13: f1=0.280, predicted=24, gold=69, saved=14/400
Document 14: f1=0.414, predicted=22, gold=36, saved=15/400
Document 15: f1=0.218, predicted=30, gold=89, saved=16/400
Document 16: f1=0.320, predicted=31, gold=69, saved=17/400
Document 17: f1=0.208, predicted=28, gold=68, sav

{'documents': 400,
 'labels': ['Activity',
  'Administration',
  'Age',
  'Area',
  'Biological_attribute',
  'Biological_structure',
  'Clinical_event',
  'Color',
  'Coreference',
  'Date',
  'Detailed_description',
  'Diagnostic_procedure',
  'Disease_disorder',
  'Distance',
  'Dosage',
  'Duration',
  'Family_history',
  'Frequency',
  'Height',
  'History',
  'Lab_value',
  'Mass',
  'Medication',
  'Nonbiological_location',
  'Occupation',
  'Other_entity',
  'Other_event',
  'Outcome',
  'Personal_background',
  'Qualitative_concept',
  'Quantitative_concept',
  'Severity',
  'Sex',
  'Shape',
  'Sign_symptom',
  'Subject',
  'Texture',
  'Therapeutic_procedure',
  'Time',
  'Volume',
  'Weight'],
 'matching': 'same label + overlapping spans + word-level subset in either direction',
 'metrics': {'precision': 0.4588192419825073,
  'recall': 0.18871318294236678,
  'f1': 0.2674313631777388},
 'counts': {'true_positive': 5036, 'predicted': 10976, 'gold': 26686}}

In [9]:
document_results_df = pd.DataFrame(document_rows)
per_label_df = pd.DataFrame(
    [
        {
            "label": label,
            "precision": values["metrics"]["precision"],
            "recall": values["metrics"]["recall"],
            "f1": values["metrics"]["f1"],
            **values["counts"],
        }
        for label, values in per_label_summary.items()
    ]
)

display(document_results_df)
display(per_label_df)


,document_index,true_positive,predicted,gold
0,0,18,28,74
1,1,7,26,55
2,2,17,28,71
3,3,11,30,42
4,4,16,27,66
...,...,...,...,...
395,395,4,27,15
396,396,7,28,46
397,397,10,26,74
398,398,19,28,72


,label,precision,recall,f1,true_positive,predicted,gold
0,Activity,0.357143,0.101010,0.157480,10,28,99
1,Administration,0.132653,0.160494,0.145251,13,98,81
2,Age,0.734082,0.980000,0.839400,392,534,400
3,Area,0.147541,0.360000,0.209302,18,122,50
4,Biological_attribute,0.000000,0.000000,0.000000,0,308,4
5,Biological_structure,0.405465,0.243892,0.304577,549,1354,2251
6,Clinical_event,0.009259,0.004908,0.006415,4,432,815
7,Color,0.000000,0.000000,0.000000,0,6,16
8,Coreference,0.000000,0.000000,0.000000,0,98,375
9,Date,0.851485,0.173038,0.287625,172,202,994


In [10]:
summary, per_label_summary, document_rows = save_evaluation_outputs(
    OUTPUT_DIR,
    list(prediction_by_index.values()),
)

OUTPUT_DIR


PosixPath('/Users/kabilan/Documents/DL Lab/Deep-learning-Lab/outputs/qwen25-maccrobat-evaluation')